In [ ]:
import time
import timm
import random
import warnings
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast

from pathlib import Path
from scipy.signal import resample_poly

warnings.filterwarnings('ignore')

# /kaggle/input/competitions/birdclef-2026/

In [ ]:
KAGGLE_PATH = Path("/kaggle/input")
DATA_PATH = KAGGLE_PATH / "competitions/birdclef-2026"
MODELS_PATH = KAGGLE_PATH / "datasets/vladimirsydor/bird-clef-2025-pretrained-models"
MODEL_PATH = MODELS_PATH / "tf_efficientnetv2_s_in21k_pretrain_from_bigXCV2_best.ckpt"
SAMPLE_SUB = DATA_PATH / 'sample_submission.csv'
TEST_DIR = DATA_PATH / 'test_soundscapes'
OUTPUT_DIR = Path('/kaggle/working')
WEIGHTS_PATH = KAGGLE_PATH / 'models/zerandr/birdclef-2026-models/pytorch/default/13/pretrained_efficientnet_secondary03_soundscapes1_bce_loss_cross_val_model'

FOLD_WEIGHTS = [
    WEIGHTS_PATH / 'best_model_fold_0.pth',
    WEIGHTS_PATH / 'best_model_fold_1.pth',
    WEIGHTS_PATH / 'best_model_fold_2.pth',
    WEIGHTS_PATH / 'best_model_fold_3.pth',
    WEIGHTS_PATH / 'best_model_fold_4.pth',
]

# Optional: per-fold weights (uniform by default)
FOLD_ENSEMBLE_WEIGHTS = [1.0, 1.0, 1.0, 1.0, 1.0]

N_CLASSES = 234
RANDOM_SEED = 42
IMG_SIZE = 224

BATCH_SIZE = 32
NUM_WORKERS = 4

SR = 32000
DURATION = 5
N_MELS = 128
N_FFT = 1024
HOP_LENGTH = 320
FMIN = 50
FMAX = 14000

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

In [ ]:
sample_sub = pd.read_csv(SAMPLE_SUB)

SPECIES_COLS = [c for c in sample_sub.columns if c != 'row_id']
NUM_CLASSES = len(SPECIES_COLS)
label2idx = {label: idx for idx, label in enumerate(SPECIES_COLS)}
idx2label = {idx: label for label, idx in label2idx.items()}

print(f'num classes : {NUM_CLASSES}')
print(f'test files  : {len(list(TEST_DIR.glob("*.ogg")))}')

In [ ]:
def fix_keys(state_dict):
    new_sd = {}
    for k, v in state_dict.items():
        k = k.replace('model.', '')
        k = k.replace('stem_conv', 'stem.conv')
        k = k.replace('stages_', 'stages.')

        new_sd[k] = v
    return new_sd


class BirdModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            # 'eca_nfnet_l0',
            'tf_efficientnetv2_s.in21k',
            pretrained=False,
            in_chans=1,
            num_classes=N_CLASSES,
        )
        ckpt = torch.load(MODEL_PATH, map_location='cpu')
        state_dict = ckpt.get('state_dict', ckpt)
        state_dict = fix_keys(state_dict)
        
        missing, unexpected = self.backbone.load_state_dict(state_dict, strict=False)
        
        print("Missing:", len(missing))
        print("Unexpected:", len(unexpected))

    def forward(self, x):
        return self.backbone(x)


model = BirdModel().to(device)

In [ ]:
mel_transform = None


def get_mel_transform(sr=SR, device='cpu'):
    global mel_transform
    if mel_transform is None:
        mel_transform = T.MelSpectrogram(
            sample_rate=SR,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH,
            n_mels=N_MELS,
            f_min=FMIN,
            f_max=FMAX,
            power=2.0,
            center=True,
            pad_mode='constant',
            norm='slaney',
            mel_scale='slaney',
        ).to(device)
    return mel_transform


def audio_to_melspec(y: np.ndarray, sr=SR, device='cpu') -> np.ndarray:
    wav = torch.from_numpy(y).unsqueeze(0).to(device)

    mel_transform = get_mel_transform(sr, device)
    mel = mel_transform(wav)

    mel_db = T.AmplitudeToDB(stype='power', top_db=None)(mel)
    # mel_db = T.AmplitudeToDB(stype='power', top_db=80.0)(mel)

    mel_db = mel_db.squeeze(0)
    mn = mel_db.min()
    mx = mel_db.max()
    mel_db = (mel_db - mn) / (mx - mn + 1e-6)

    return mel_db.cpu().numpy().astype(np.float32)


def load_exact_window(filepath, start_sec, end_sec,
                      sr=SR, duration=DURATION) -> np.ndarray:
    target_len = int(sr * duration)

    if start_sec is None and end_sec is None:
        start_sec, end_sec = 0.0, float(duration)
    elif start_sec is None:
        start_sec = max(0.0, float(end_sec) - float(duration))
    elif end_sec is None:
        end_sec = float(start_sec) + float(duration)
    if float(end_sec) <= float(start_sec):
        end_sec = float(start_sec) + float(duration)

    try:
        wav, file_sr = torchaudio.load(str(filepath))
        wav = wav.mean(dim=0)
        start = int(start_sec * file_sr) if start_sec is not None else 0
        end = int(end_sec * file_sr) if end_sec is not None else start + int(duration * file_sr)

        if end <= start:
            end = start + int(duration * file_sr)
        wav = wav[start:end]

        if file_sr != sr:
            wav = F.resample(wav, file_sr, sr)

        y = wav.numpy()
    except Exception as e:
        y = np.zeros(target_len, dtype=np.float32)

    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    elif len(y) > target_len:
        y = y[:target_len]
    return y.astype(np.float32)


class SoundscapeDataset(Dataset):
    """Loads test soundscapes and returns raw waveforms for GPU mel."""
    def __init__(self, sample_sub, audio_dir):
        self.items = []
        self.audio_dir = Path(audio_dir)

        for _, row in sample_sub.iterrows():
            parts    = row['row_id'].split('_')
            end_time = int(parts[-1])
            start_time = end_time - DURATION
            stem     = '_'.join(parts[:-1])
            filepath = self.audio_dir / f'{stem}.ogg'
            self.items.append((row['row_id'], filepath, start_time, end_time))
        exists = sum(1 for _, fp, _, _ in self.items if fp.exists())
        print(f'windows: {len(self.items):,}  |  files found: {exists:,}')

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        row_id, filepath, start_sec, end_sec = self.items[idx]

        y = load_exact_window(str(filepath), start_sec=start_sec, end_sec=end_sec, duration=DURATION)

        mel = audio_to_melspec(y)

        mel_tensor = torch.from_numpy(mel).unsqueeze(0)
        mel_tensor = F.interpolate(
            mel_tensor.unsqueeze(0), size=(IMG_SIZE, IMG_SIZE),
            mode='bilinear', align_corners=False
        ).squeeze(0)

        return mel_tensor, row_id

In [ ]:
# ── Ensemble inference over all folds ──────────────────────────────────────
# Build dataloader once (shared across all folds)
inf_ds     = SoundscapeDataset(sample_sub, TEST_DIR)
inf_loader = DataLoader(inf_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

print(f'inference batches : {len(inf_loader)}')
print(f'ensemble folds    : {len(FOLD_WEIGHTS)}')

# Accumulate weighted sum of probabilities
fold_weight_total = sum(FOLD_ENSEMBLE_WEIGHTS)
ensemble_probs = None   # will be np.ndarray after first fold
all_row_ids    = None

t_total = time.time()

for fold_idx, (weights_path, fold_w) in enumerate(zip(FOLD_WEIGHTS, FOLD_ENSEMBLE_WEIGHTS)):
    print(f'\n── Fold {fold_idx}  (weight={fold_w})  {weights_path.name}')
    
    # Load fold weights
    model.load_state_dict(torch.load(weights_path, map_location=device))
    model.eval()
    torch.cuda.empty_cache()

    fold_row_ids = []
    fold_probs   = []
    t0           = time.time()

    with torch.no_grad():
        with autocast():
            for batch_idx, (images, row_ids) in enumerate(inf_loader):
                images = images.to(device, non_blocking=True)
                with autocast():
                    logits = model(images)
                probs = torch.sigmoid(logits).float().cpu().numpy()

                fold_row_ids.extend(row_ids)
                fold_probs.append(probs)

                if batch_idx % 20 == 0:
                    pct = (batch_idx + 1) / len(inf_loader) * 100
                    print(f'  {batch_idx+1}/{len(inf_loader)} ({pct:.0f}%)  '
                          f'elapsed={time.time()-t0:.0f}s')

    fold_probs = np.vstack(fold_probs)
    print(f'  fold {fold_idx} done in {time.time()-t0:.0f}s  shape: {fold_probs.shape}')

    # Accumulate
    if ensemble_probs is None:
        ensemble_probs = fold_probs * fold_w
        all_row_ids    = fold_row_ids
    else:
        assert all_row_ids == fold_row_ids, 'row_id mismatch between folds!'
        ensemble_probs += fold_probs * fold_w

torch.cuda.empty_cache()

# Average
all_probs = ensemble_probs / fold_weight_total
print(f'\nTotal ensemble time: {time.time()-t_total:.0f}s')
print(f'Final probs shape  : {all_probs.shape}')
print(f'Prob range         : [{all_probs.min():.4f}, {all_probs.max():.4f}]')

In [ ]:
sub_df = pd.DataFrame(all_probs, columns=SPECIES_COLS)
sub_df.insert(0, 'row_id', all_row_ids)
sub_df = sample_sub[['row_id']].merge(sub_df, on='row_id', how='left')
sub_df[SPECIES_COLS] = sub_df[SPECIES_COLS].fillna(0.0)

assert len(sub_df) == len(sample_sub), 'row mismatch'
assert list(sub_df.columns) == list(sample_sub.columns), 'column mismatch'
assert sub_df[SPECIES_COLS].isnull().sum().sum() == 0, 'nulls found'

print('=== sanity checks ===')
print(f'rows    : {len(sub_df):,}')
print(f'nulls   : {sub_df[SPECIES_COLS].isnull().sum().sum()}')
print(f'min prob: {sub_df[SPECIES_COLS].min().min():.4f}')
print(f'max prob: {sub_df[SPECIES_COLS].max().max():.4f}')
print('all checks passed ✅')

sub_path = OUTPUT_DIR / 'submission.csv'
sub_df.to_csv(sub_path, index=False)
print(f'\nsaved → {sub_path}')
sub_df.head()